In [ ]:
"""
Fact-checking vietnamita 
Peso final: ~130-150 MB | Sin GPU requerida
pip install torch transformers scikit-learn kagglehub
"""

import json, random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ── CONFIG ───────────────────────────────────────────────────────────────────
CONFIG = {
    "model_name":     "distilbert-base-multilingual-cased",
    "max_length":     64,
    "batch_size":     32,
    "epochs":         2,
    "learning_rate":  3e-5,
    "weight_decay":   0.01,
    "warmup_ratio":   0.1,
    "seed":           42,
    "output_dir":     "./outputs/mvp-fact-checking",
    "evidence_words": 40,
    "max_samples":    2000,
}

LABEL2ID = {"SUPPORTED": 0, "REFUTED": 1}
ID2LABEL = {0: "SUPPORTED", 1: "REFUTED"}

# ── SEMILLA ──────────────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# ── CARGA DEL DATASET ────────────────────────────────────────────────────────
def load_dataset_from_dir(dataset_dir: str):
    dataset_dir = Path(dataset_dir)
    samples = []

    all_files = list(dataset_dir.rglob("*.json"))
    files     = [f for f in all_files if "HIGH_CONFIDENCE" in f.name] or all_files
    print(f"Archivos: {[f.name for f in files]}")

    for fpath in files:
        try:
            content = json.loads(fpath.read_text(encoding="utf-8"))
            for item in (content if isinstance(content, list) else [content]):
                label    = item.get("label", "").upper().strip()
                if label not in LABEL2ID:
                    continue
                claim    = item.get("claim", "")
                evidence = item.get("contexts", item.get("evidence", ""))
                if isinstance(evidence, list):
                    evidence = " ".join(e if isinstance(e, str) else " ".join(e) for e in evidence)
                if claim and evidence:
                    samples.append({"claim": claim.strip(), "evidence": evidence.strip(), "label": label})
        except Exception as e:
            print(f"  Error {fpath.name}: {e}")

    random.shuffle(samples)
    if CONFIG["max_samples"]:
        samples = samples[:CONFIG["max_samples"]]

    counts = {}
    for s in samples:
        counts[s["label"]] = counts.get(s["label"], 0) + 1
    print(f"Total: {len(samples)} | {counts}")
    return samples

def split_dataset(samples):
    n = len(samples)
    return samples[:int(n*0.8)], samples[int(n*0.8):int(n*0.9)], samples[int(n*0.9):]

# ── DATASET ──────────────────────────────────────────────────────────────────
class FactCheckDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.samples   = samples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item     = self.samples[idx]
        evidence = " ".join(item["evidence"].split()[:CONFIG["evidence_words"]])
        enc      = self.tokenizer(
            item["claim"], evidence,
            max_length=CONFIG["max_length"],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(LABEL2ID[item["label"]], dtype=torch.long),
        }

# ── TRAIN / EVAL ─────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, preds_all, labels_all = 0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            out  = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
            if training:
                optimizer.zero_grad()
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += out.loss.item()
            preds_all.extend(out.logits.argmax(-1).numpy())
            labels_all.extend(batch["label"].numpy())

    return total_loss / len(loader), accuracy_score(labels_all, preds_all), f1_score(labels_all, preds_all, average="macro"), preds_all, labels_all

# ── INFERENCIA ───────────────────────────────────────────────────────────────
def predict(claim: str, evidence: str, model, tokenizer) -> dict:
    """Output formato torneo: {'predicted_label': 'SUPPORTED' | 'REFUTED'}"""
    model.eval()
    enc = tokenizer(claim, evidence, max_length=CONFIG["max_length"], padding="max_length", truncation=True, return_tensors="pt")
    with torch.no_grad():
        pred = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits.argmax(-1).item()
    return {"predicted_label": ID2LABEL[pred]}

# ── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    set_seed(CONFIG["seed"])
    print("Device: CPU")

    import kagglehub
    dataset_dir = kagglehub.dataset_download("haisemei/fact-checking-dataset-label")

    samples = load_dataset_from_dir(dataset_dir)
    if not samples:
        raise ValueError("No se cargaron muestras.")

    train_data, val_data, test_data = split_dataset(samples)
    print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

    print(f"\nCargando {CONFIG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    model     = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"], num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
    )

    train_loader = DataLoader(FactCheckDataset(train_data, tokenizer), batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader   = DataLoader(FactCheckDataset(val_data,   tokenizer), batch_size=CONFIG["batch_size"])
    test_loader  = DataLoader(FactCheckDataset(test_data,  tokenizer), batch_size=CONFIG["batch_size"])

    optimizer   = AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    total_steps = len(train_loader) * CONFIG["epochs"]
    scheduler   = get_linear_schedule_with_warmup(optimizer, int(total_steps * CONFIG["warmup_ratio"]), total_steps)

    output_dir   = Path(CONFIG["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    best_val_acc = 0

    print("\n" + "="*50)
    for epoch in range(1, CONFIG["epochs"] + 1):
        tr_loss, tr_acc, _,      _, _ = run_epoch(model, train_loader, optimizer, scheduler)
        vl_loss, vl_acc, vl_f1,  _, _ = run_epoch(model, val_loader)
        print(f"Época {epoch}/{CONFIG['epochs']} | Train Loss {tr_loss:.4f} Acc {tr_acc:.4f} | Val Loss {vl_loss:.4f} Acc {vl_acc:.4f} F1 {vl_f1:.4f}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            model.save_pretrained(output_dir / "best_model")
            tokenizer.save_pretrained(output_dir / "best_model")
            print(f"  ✓ Guardado (val_acc={vl_acc:.4f})")

    # ── Evaluación final ─────────────────────────────────────────────────────
    print("\n" + "="*50)
    _, test_acc, test_f1, test_preds, test_labels = run_epoch(model, test_loader)
    print(f"Test Accuracy: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(classification_report(test_labels, test_preds, target_names=["SUPPORTED", "REFUTED"]))

    # ── Cuantización → reduce ~4x el peso ────────────────────────────────────
    print("\nCuantizando modelo...")
    best_model = AutoModelForSequenceClassification.from_pretrained(output_dir / "best_model")
    best_model.eval()
    quantized  = torch.quantization.quantize_dynamic(best_model, {torch.nn.Linear}, dtype=torch.qint8)

    quant_path = output_dir / "model_quantized"
    quant_path.mkdir(parents=True, exist_ok=True)
    torch.save(quantized.state_dict(), quant_path / "model.pt")
    tokenizer.save_pretrained(quant_path)

    # ── Tamaños ───────────────────────────────────────────────────────────────
    size_original  = sum(f.stat().st_size for f in (output_dir / "best_model").rglob("*") if f.is_file())
    size_quantized = sum(f.stat().st_size for f in quant_path.rglob("*") if f.is_file())
    print(f"Original:   {size_original  / 1e6:.1f} MB")
    print(f"Cuantizado: {size_quantized / 1e6:.1f} MB")

    # ── Ejemplo de inferencia con modelo cuantizado ───────────────────────────
    print("\n" + "="*50)
    res = predict("Hà Nội là thủ đô của Việt Nam.", "Hà Nội là thủ đô và là thành phố lớn nhất của Việt Nam.", quantized, tokenizer)
    print(f"Ejemplo inferencia: {res}")
    print(f"\nModelo listo en: {quant_path}")

if __name__ == "__main__":
    main()

Device: CPU
Using Colab cache for faster access to the 'fact-checking-dataset-label' dataset.
Archivos: ['HIGH_CONFIDENCE_test_dataset.json', 'HIGH_CONFIDENCE_converted_dataset.json', 'HIGH_CONFIDENCE_validation_dataset.json', 'HIGH_CONFIDENCE_train_dataset.json']
Total: 2000 | {'SUPPORTED': 1336, 'REFUTED': 664}
Train: 1600 | Val: 200 | Test: 200

Cargando distilbert-base-multilingual-cased...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Época 1/2 | Train Loss 0.6263 Acc 0.6694 | Val Loss 0.6674 Acc 0.6250 F1 0.3846


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Guardado (val_acc=0.6500)

Test Accuracy: 0.6700 | F1: 0.4528
              precision    recall  f1-score   support

   SUPPORTED       0.67      0.98      0.80       133
     REFUTED       0.57      0.06      0.11        67

    accuracy                           0.67       200
   macro avg       0.62      0.52      0.45       200
weighted avg       0.64      0.67      0.57       200


Cuantizando modelo...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_7960/267174815.py:190: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized  = torch.quantization.quantize_dynamic(best_model, {torch.nn.Linear}, dtype=torch.qint8)


Original:   544.2 MB
Cuantizado: 415.1 MB

Ejemplo inferencia: {'predicted_label': 'SUPPORTED'}

Modelo listo en: outputs/mvp-fact-checking/model_quantized


In [8]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "distilbert-base-multilingual-cased"
)

config.save_pretrained(
    "outputs/mvp-fact-checking/model_quantized"
)

print("config.json guardado")

config.json guardado


In [9]:
import shutil

shutil.make_archive(
    "model_quantized",
    "zip",
    "outputs/mvp-fact-checking/model_quantized"
)

'/content/model_quantized.zip'